In [1]:
import numpy as np
import torch
import torch.utils.data as data
import torch.nn as nn
import sys
import warnings
from itertools import product

## NoteBook environment specific (has to be configured on user side)

In [3]:
# import github repo into notebook file directory, might require fixing path
!rm -r /content/Architectural-Biases-in-Time-Series-Anomaly-Detection
!git clone https://github.com/KirillVishnyakov/Architectural-Biases-in-Time-Series-Anomaly-Detection
!git -C /content/Architectural-Biases-in-Time-Series-Anomaly-Detection log --oneline -1

# path to the github repo
root_dir = "/content/Architectural-Biases-in-Time-Series-Anomaly-Detection"
sys.path.append(root_dir)

import utils.config as config
# arg1: path to the cats dataset, arg2: path to the checkpoint folder
config.init("/content/data.csv", root_dir + "/checkpoint_dir")

Cloning into 'Architectural-Biases-in-Time-Series-Anomaly-Detection'...
remote: Enumerating objects: 432, done.
remote: Counting objects: 100% (76/76), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 432 (delta 37), reused 56 (delta 19), pack-reused 356 (from 1)
Receiving objects: 100% (432/432), 119.01 MiB | 30.28 MiB/s, done.
Resolving deltas: 100% (223/223), done.
70674e6 (HEAD -> main, origin/main, origin/HEAD) typo


### Setup the environment to have access to required repo functions

In [4]:
warnings.filterwarnings("ignore")
from models.transformer_encoder_forecaster import patch_transformer
import dataset
import train

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.__version__)
print(device)

2.10.0+cu128
cuda


### Random states for reproducibility

In [5]:
seed = 432
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

### Tuning Helper

In [6]:
def tune_over_grid(hyperparam_grid, train_dataset, test_dataset, epochs = 5):
    results = []
    #d_model, nhead, dropout, num_features, forecast_horizon, num_blocks, lookback_window, lr
    for lr, lookback_window, d_model, num_blocks in product(
        hyperparam_grid["lr"],
        hyperparam_grid["lookback_window"],
        hyperparam_grid["d_model"],
        hyperparam_grid["num_blocks"]
    ):
        model = patch_transformer(
                    lookback_window = lookback_window, forecast_horizon = 4, d_model = d_model,
                    nhead = 8, dropout = 0.0, num_features = 17, num_blocks = num_blocks
                ).to(device)
        model = torch.compile(model)

        name = f"lr_{lr}_lw_{lookback_window}_d_model_{d_model}_num_blocks_{num_blocks}"

        weights, val_loss, train_loss = \
        train.fit_forecaster(device, model, name, train_dataset, test_dataset, lr = lr, batch_size = 512, num_epochs = epochs)

        results.append({
            "val_loss": val_loss,
            "name": name,
            "weights": weights
        })

    results.sort(key=lambda x: x['val_loss'])
    return results


In [7]:
train_dataset_256 = dataset.forecasting_Dataset(device = device, window_size = 256, horizon = 4, start = 0, end = 800000)
test_dataset_256 = dataset.forecasting_Dataset(device = device, window_size = 256, horizon = 4, scaler = train_dataset_256.scaler, start = 800000, end = 1000000, train = False)

### Tune over lr

In [ ]:
hyperparam_grid = \
{ "lr": [0.0006, 0.001, 0.0012],
  "lookback_window": [256],
  "d_model": [256],
  "num_blocks": [2]
}

results = tune_over_grid(hyperparam_grid, train_dataset_256, test_dataset_256, epochs = 5)
results_best = results[0]
print(f"best model: {results_best['name']}, {results_best['val_loss']}")

32
LR Scheduler: 781 warmup steps, 7810 total steps
|lr_256_lw_256_d_model_256_num_blocks_2| train = 0.1395 | test= 0.0213 | LR: 5.82e-04
|lr_256_lw_256_d_model_256_num_blocks_2| train = 0.0219 | test= 0.0207 | LR: 4.51e-04
|lr_256_lw_256_d_model_256_num_blocks_2| train = 0.0212 | test= 0.0205 | LR: 2.51e-04
|lr_256_lw_256_d_model_256_num_blocks_2| train = 0.0209 | test= 0.0203 | LR: 7.55e-05
|lr_256_lw_256_d_model_256_num_blocks_2| train = 0.0207 | test= 0.0203 | LR: 6.00e-06
32
LR Scheduler: 781 warmup steps, 7810 total steps
|lr_256_lw_256_d_model_256_num_blocks_2| train = 0.1659 | test= 0.0219 | LR: 9.70e-04
|lr_256_lw_256_d_model_256_num_blocks_2| train = 0.0218 | test= 0.0208 | LR: 7.53e-04
|lr_256_lw_256_d_model_256_num_blocks_2| train = 0.0212 | test= 0.0204 | LR: 4.19e-04
|lr_256_lw_256_d_model_256_num_blocks_2| train = 0.0209 | test= 0.0203 | LR: 1.26e-04
|lr_256_lw_256_d_model_256_num_blocks_2| train = 0.0207 | test= 0.0202 | LR: 1.00e-05
32
LR Scheduler: 781 warmup steps, 7

#### TODO: tune over lookback window

### Tune over d_model

In [ ]:
hyperparam_grid = \
{ "lr": [0.001],
  "lookback_window": [256],
  "d_model": [128, 384],
  "num_blocks": [2]
}

results = tune_over_grid(hyperparam_grid, train_dataset_256, test_dataset_256, epochs = 5)
results_best = results[0]
print(f"best model: {results_best['name']}, {results_best['val_loss']}")

32
LR Scheduler: 781 warmup steps, 7810 total steps
|lr_0.001_lw_256_d_model_128_num_blocks_2| train = 0.0805 | test= 0.0211 | LR: 9.70e-04
|lr_0.001_lw_256_d_model_128_num_blocks_2| train = 0.0218 | test= 0.0207 | LR: 7.53e-04
|lr_0.001_lw_256_d_model_128_num_blocks_2| train = 0.0212 | test= 0.0205 | LR: 4.19e-04
|lr_0.001_lw_256_d_model_128_num_blocks_2| train = 0.0209 | test= 0.0203 | LR: 1.26e-04
|lr_0.001_lw_256_d_model_128_num_blocks_2| train = 0.0207 | test= 0.0203 | LR: 1.00e-05
32
LR Scheduler: 781 warmup steps, 7810 total steps
|lr_0.001_lw_256_d_model_384_num_blocks_2| train = 0.2186 | test= 0.0219 | LR: 9.70e-04
|lr_0.001_lw_256_d_model_384_num_blocks_2| train = 0.0221 | test= 0.0214 | LR: 7.53e-04
|lr_0.001_lw_256_d_model_384_num_blocks_2| train = 0.0213 | test= 0.0205 | LR: 4.19e-04
|lr_0.001_lw_256_d_model_384_num_blocks_2| train = 0.0209 | test= 0.0203 | LR: 1.26e-04
|lr_0.001_lw_256_d_model_384_num_blocks_2| train = 0.0207 | test= 0.0202 | LR: 1.00e-05
best model: lr_0

#### TODO: tune over num_blocks

### Train best model so far on full dataset with more epochs

In [8]:
best_transformer_model = \
patch_transformer(lookback_window = 256, forecast_horizon = 4, d_model = 256, nhead = 8, dropout = 0.0, num_features = 17, num_blocks = 1).to(device)

best_transformer_model = torch.compile(best_transformer_model)

best_model_wts, best_loss, avg_train_loss = \
train.fit_forecaster(device, best_transformer_model, "transformer_encoder", train_dataset_256, test_dataset_256, lr = 0.001, batch_size = 512, num_epochs = 10)

32
LR Scheduler: 781 warmup steps, 15620 total steps
|transformer_encoder| train = 0.1916 | test= 0.0219 | LR: 9.93e-04
|transformer_encoder| train = 0.0221 | test= 0.0210 | LR: 9.40e-04
|transformer_encoder| train = 0.0220 | test= 0.0203 | LR: 8.40e-04
|transformer_encoder| train = 0.0212 | test= 0.0203 | LR: 7.04e-04
|transformer_encoder| train = 0.0211 | test= 0.0203 | LR: 5.46e-04
|transformer_encoder| train = 0.0209 | test= 0.0202 | LR: 3.83e-04
|transformer_encoder| train = 0.0208 | test= 0.0201 | LR: 2.34e-04
|transformer_encoder| train = 0.0207 | test= 0.0200 | LR: 1.14e-04
|transformer_encoder| train = 0.0206 | test= 0.0200 | LR: 3.68e-05
|transformer_encoder| train = 0.0205 | test= 0.0200 | LR: 1.00e-05


In [9]:
torch.save(best_transformer_model._orig_mod.state_dict(), "transformer_forecaster_weights.pt")